# 00. Imports & Upload DB

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# AutoWoE из LightAutoML
#from lightautoml.addons.autowoe import AutoWoE
from autowoe import AutoWoE, ReportDeco

### ---- GINI Time Trend


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


def gini_binary(y_true, y_pred):
    """Считает Gini через AUC"""
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc - 1


def compute_confidence_interval(gini, n, alpha=0.05):
    """Простой нормальный аппрокс. ДИ"""
    se = np.sqrt((gini * (1 - gini)) / n) if n > 0 else 0
    z = norm.ppf(1 - alpha / 2)
    return gini - z * se, gini + z * se


def plot_gini_dynamics_binary(df1, observed_col, predicted_col, date_col,
                              freq='M', min_observations=100,
                              alpha_95=0.05, alpha_99=0.01,
                              figsize=(14, 8), title=None):
    """
    Динамика Gini (бинарная классификация) с доверительными интервалами
    """

    freq = str(freq).upper()

    data = df1.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    freq_map = {
        'D': 'D',
        'W': 'W',
        'M': 'M',
        'Q': 'Q',
        'Y': 'Y'
    }

    period_freq = freq_map.get(freq, 'M')
    grouped = data.groupby(data[date_col].dt.to_period(period_freq), sort=True)
    
    results = []

    for period, group_data in grouped:
        if len(group_data) >= min_observations:
            try:
                gini_model = gini_binary(group_data[observed_col], group_data[predicted_col])
                n_obs = len(group_data)

                ci_95_low, ci_95_high = compute_confidence_interval(gini_model, n_obs, alpha_95)
                ci_99_low, ci_99_high = compute_confidence_interval(gini_model, n_obs, alpha_99)

                results.append({
                    "period": period,
                    "gini": gini_model,
                    "n_observations": n_obs,
                    "ci_95_low": ci_95_low,
                    "ci_95_high": ci_95_high,
                    "ci_99_low": ci_99_low,
                    "ci_99_high": ci_99_high,
                })
            except Exception as e:
                print(f"Ошибка в периоде {period}: {e}")
                continue

    if not results:
        print("Нет данных для графика")
        return None, None

    results_df1 = pd.DataFrame(results).sort_values('period')
    x_positions = np.arange(len(results_df1))

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    # Столбцы = число наблюдений
    ax2.bar(x_positions, results_df1['n_observations'],
            alpha=0.3, color="#2ca02c", label="Количество наблюдений", width=0.6)

    # 99% ДИ
    ax1.fill_between(x_positions, results_df1['ci_99_low'], results_df1['ci_99_high'],
                     alpha=0.3, color="#ff4444", label="99% ДИ")

    # 95% ДИ
    ax1.fill_between(x_positions, results_df1['ci_95_low'], results_df1['ci_95_high'],
                     alpha=0.5, color="#ffcc00", label="95% ДИ")

    # Линия Gini
    ax1.plot(x_positions, results_df1['gini'], color='black',
             linewidth=2, marker='o', markersize=6, label="Gini")

    # подписи на точках
    for x_pos, (_, row) in zip(x_positions, results_df1.iterrows()):
        ax1.annotate(f"{row['gini']:.3f}",
                     (x_pos, row['gini']),
                     textcoords="offset points", xytext=(0, 10),
                     ha='center', fontsize=9, fontweight='bold')

    # Пороговые линии
    ax1.axhline(y=gini_binary(data[observed_col], data[predicted_col]), color='purple', linestyle='--', alpha=0.7, label='Среднее Gini')
    ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Порог 20%')
    ax1.axhline(y=0.3, color='green', linestyle='--', alpha=0.7, label='Порог 30%')

    # Оси
    ax1.set_xlabel("Период", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Gini", fontsize=12, fontweight='bold', color="#1f77b4")
    ax2.set_ylabel("Количество наблюдений", fontsize=12, fontweight='bold', color="#2ca02c")

    ax1.tick_params(axis='y', labelcolor="#1f77b4")
    ax2.tick_params(axis='y', labelcolor="#2ca02c")

    # Формат периодов без визуального сдвига к следующему месяцу/кварталу
    def _format_period(period):
        if freq == 'Q':
            return f"{period.year}-Q{period.quarter}"
        if freq == 'Y':
            return f"{period.year}"
        if freq == 'M':
            return period.start_time.strftime('%Y-%m')
        if freq == 'W':
            return f"{period.start_time:%Y-%m-%d}\n{period.end_time:%Y-%m-%d}"
        return period.start_time.strftime('%Y-%m-%d')

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([_format_period(period) for period in results_df1['period']])
    ax1.set_xlim(-0.5, len(results_df1) - 0.5)

    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    if title is None:
        title = "Динамика Gini (бинарная классификация)"
    ax1.set_title(title, fontsize=14, fontweight='bold', pad=20)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               loc='upper left', frameon=True, fancybox=True, shadow=True)

    ax1.grid(True, alpha=0.3)
    plt.tight_layout()

    print(f"Построен график для {len(results_df1)} периодов")
    print(f"Gini: {gini_binary(data[observed_col], data[predicted_col]):.3f}")
    print(f"Мин Gini: {results_df1['gini'].min():.3f}")
    print(f"Макс Gini: {results_df1['gini'].max():.3f}")
    print(f"Среднее кол-во наблюдений: {results_df1['n_observations'].mean():.0f}")

    return fig, ax1


### --- Upload DB

In [2]:
main_db_path = r"E:\PubPricing\01_Retail\03_PPI\01_Modelling\01_HT\2026m03\00_DB\radar_property_w_tags_ness_fields_prom_v5_2026_01_not_mort_pt_23_25.csv"

HH_main_df = pd.read_csv(
    main_db_path,
    low_memory=False,
    sep='^',
    encoding='cp1251',
    decimal=','
)
HH_main_df.set_index('CO_KEY', inplace=True)

print(HH_main_df.shape)
HH_main_df.head()

(3197955, 778)


,CONTRACT_ID,OBJECT_TYPE,POLICY_DATE,POLICY_START_DATE,POLICY_END_DATE,POLICY_CANCEL_DATE,EXPOSURE_POLICY_PERIOD,ACC_CNT_TOTAL,ACC_CNT_FIRE,ACC_CNT_WATER,...,TAG_39247,TAG_39248,MULTI_CONTRACT_IND,INIT_CONTRACT_NUMBER,INIT_POLICY_START_DATE,INIT_POLICY_END_DATE,INIT_POLICY_CANCEL_DATE,DIFF_TO_INIT_START_DATE_DAYS,DIFF_TO_INIT_START_DATE_MONTHS,DIFF_TO_INIT_START_DATE_YEARS
CO_KEY,,,,,,,,,,,,,,,,,,,,,
9AE50050560121AD11EE996DB6CC239BКвартира,9AE50050560121AD11EE996DB6CC239B,Квартира,2023-12-13 00:00:00.0,2023-12-18 00:00:00.0,2024-01-17 00:00:00.0,NaN,0.084932,0,0,0,...,0,0,1,007WS5654972327,2023-08-18 00:00:00.0,2023-09-17 00:00:00.0,NaN,122,4,0
8289005056012C8811EDB9B8F3045F35Квартира,8289005056012C8811EDB9B8F3045F35,Квартира,2023-03-03 00:00:00.0,2023-03-11 00:00:00.0,2024-03-10 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2023-03-11 00:00:00.0,2024-03-10 00:00:00.0,NaN,0,0,0
8207005056012C8811ED0C3CA0BDF684Квартира,8207005056012C8811ED0C3CA0BDF684,Квартира,2022-07-25 00:00:00.0,2022-08-08 00:00:00.0,2023-08-07 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2022-08-08 00:00:00.0,2023-08-07 00:00:00.0,NaN,0,0,0
9B0A0050560A61CE11EF64EC36DCAF9CКвартира,9B0A0050560A61CE11EF64EC36DCAF9C,Квартира,2024-08-27 00:00:00.0,2024-08-28 00:00:00.0,2025-08-27 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2024-08-28 00:00:00.0,2025-08-27 00:00:00.0,NaN,0,0,0
9C080050560121AD11F0508E5149A42EКвартира,9C080050560121AD11F0508E5149A42E,Квартира,2025-06-24 00:00:00.0,2025-07-01 00:00:00.0,2025-07-31 00:00:00.0,NaN,0.084932,0,0,0,...,0,0,1,007WS4700246911,2025-01-29 00:00:00.0,2025-02-28 00:00:00.0,NaN,153,5,0


### --- Target & Time setup

In [3]:
TARGET_NAME       = 'ACC_CNT_WATER_BOOL'
TIME_TREND_FACTOR = 'POLICY_END_DATE'

HH_main_df[TIME_TREND_FACTOR] = pd.to_datetime(HH_main_df[TIME_TREND_FACTOR])

print('Target rate:', HH_main_df[TARGET_NAME].mean().round(4))
print('Date range: ', HH_main_df[TIME_TREND_FACTOR].min(), '→', HH_main_df[TIME_TREND_FACTOR].max())

Target rate: 0.0069
Date range:  2023-01-01 00:00:00 → 2025-12-31 00:00:00


# 01. Feature short list

In [5]:
# Загружаем short list признаков (аналогично оригинальному ноутбуку)
FeatShortList = list(set(
    pd.read_csv(
        r'FSA_FeatTruncList02.csv',
        low_memory=False, sep=';', encoding='cp1251', decimal=','
    )['NAME']
))

# Дополнительные продуктовые признаки
feat_list_internal = [
    'PRODUCT_OBJECT_TYPE_HOUSE',
    'PRODUCT_OBJECT_TYPE_FLAT',
    'PRODUCT_OBJECT_TYPE_OTHER',
]
FeatShortList += feat_list_internal

# Оставляем только те, что реально есть в данных
FeatShortList = [f for f in FeatShortList if f in HH_main_df.columns]
print(f'Признаков в short list: {len(FeatShortList)}')

Признаков в short list: 363


# WhiteBox Preset

### --- Train / Test split

In [6]:
RANDOM_STATE = 28
TEST_SIZE    = 0.4

train_data, test_data = train_test_split(
    HH_main_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=HH_main_df[TARGET_NAME]
)

print(f'Train: {train_data.shape},  Test: {test_data.shape}')
print(f'Target rate — Train: {train_data[TARGET_NAME].mean():.4f},  Test: {test_data[TARGET_NAME].mean():.4f}')

Train: (1918773, 778),  Test: (1279182, 778)
Target rate — Train: 0.0069,  Test: 0.0069


In [7]:
from lightautoml.automl.presets.whitebox_presets import WhiteBoxPreset
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDecoWhitebox

'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


In [9]:
# Setup binary classification task  
task = Task('binary')

# Create WhiteBoxPreset with report generation enabled
automl = WhiteBoxPreset(
    task=task,
    timeout=3600*11,
    cpu_limit=40,
    general_params={"report": True},  # Enable report for Gini and scorecard
    whitebox_params={  
        'default_params': {
            'report': True,  # Enable detailed reporting  
            'oof_woe': True,  # Use OOF encoding for WoE  
            'n_folds': 7  # Number of folds for internal CV  
        }  
    }  
)  
roles = {
    'target': TARGET_NAME,
    #'weights': WEIGHT_NAME,
    #'category': [...],
    'drop': [col for col in train_data.columns if (col not in FeatShortList) and (col not in [TARGET_NAME])] # Exclude non-tag columns
             
}

In [10]:
# Wrap with report decorator for HTML scorecard
report_automl = ReportDecoWhitebox(
    output_path="./report",
    report_file_name="autowoe_report.html"
)(automl)

In [11]:
# Train model (internally calls autowoe.fit)  
oof_pred = report_automl.fit_predict(  
    train_data,  
    roles=roles, 
    #valid_data=test_data,  # Optional validation data
    verbose=4
)

[11:46:07] Stdout logging level is DEBUG.
[11:46:07] Task: binary

[11:46:07] Start automl preset with listed constraints:
[11:46:07] - time: 39600.00 seconds
[11:46:07] - CPU: 40 cores
[11:46:07] - memory: 16 GB

[11:46:07] Train data shape: (1918773, 778)

[11:47:04] Feats was rejected during automatic roles guess: []
[11:47:20] Layer 1 train process start. Time left 39527.41 secs
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 10474, number of negative: 1524544
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.454975 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 32979
[LightGBM] [Info] Number of data points in the tr

AttributeError: 'WhiteBoxPreset' object has no attribute 'get_feature_scores'

In [ ]:
# Get test predictions  
#test_pred = automl.predict(test_data)

In [ ]:
# Access the underlying AutoWoE model  
#whitebox_model = automl.whitebox

In [ ]:
# # If report is enabled, the model is wrapped in ReportDeco  
# if hasattr(whitebox_model, 'model'):  
#     # Access the actual AutoWoE model  
#     actual_model = whitebox_model.model  
#     scorecard_data = actual_model.get_scorecard()  
# else:  
#     # Direct access if no report wrapper  
#     scorecard_data = whitebox_model.get_scorecard()  
  
# # Convert to DataFrame  
# scorecard_df = pd.DataFrame(scorecard_data, columns=['Variable', 'Value', 'WOE', 'COEF', 'POINTS'])
# # Save scorecard  
# scorecard_df.to_csv('model_scorecard.csv', index=False, sep=';', encoding = 'cp1251', decimal =',')  
# print(f"Scorecard saved with {len(scorecard_df)} rows")


In [ ]:
joblib.dump(automl, 'HT_HH_PROB_WD_AUTOWOE_model.pkl')
print('Модель сохранена: HT_HH_PROB_WD_AUTOWOE_model.pkl')